# SMS Spam Classification Using NLP

##  Part 1: Introduction

### Student Information
Aviv M 1886

### AI Prompts and Additional Resources

- "ill do stemming. give me the import line and the function name. ill go implement it and ull check me." — asked for the `nltk.stem.PorterStemmer` import and a function name to implement stemming myself.

### Problem and Dataset
The goal of this project is to perform a binary classification of SMS messages and predict whether each message is Spam or Ham. For this project, I'll use the following dataset: "Spam SMS Classification Using NLP", from Kaggle, which contains labeled SMS messages classified as Spam or Ham. After loading the data, I'll perform text Feature Engineering, and then implement the Naive Bayes algorithm to classify the messages. 

### Dataset Loading

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/Spam_SMS.csv")  # load the full dataset from CSV

# split ONCE into train/test (80% / 20%)
# random_state=42 -> same split every time we run this
# stratify=df["Class"] -> keeps the same ham/spam ratio in both train and test (dividing proportionally according to "Class" column)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["Class"]
)

# Making the Class to be most right column for better visualization in the dataframe
train_df = train_df[["Message", "Class"]]
test_df = test_df[["Message", "Class"]]

print("Train set shape:", train_df.shape)
print("Test set shape:", test_df.shape)

Train set shape: (4459, 2)
Test set shape: (1115, 2)


In [2]:
train_df.head()

,Message,Class
82,Ok i am on the way to home hi hi,ham
1328,Ur balance is now £500. Ur next question is: W...,spam
577,I'm tired of arguing with you about this week ...,ham
656,Tell them the drug dealer's getting impatient,ham
317,Hmmm... Guess we can go 4 kb n power yoga... H...,ham


In [3]:
test_df.head()

,Message,Class
3625,No message..no responce..what happend?,ham
4570,At WHAT TIME should i come tomorrow,ham
1192,Come to my home for one last time i wont do an...,ham
673,Get ur 1st RINGTONE FREE NOW! Reply to this ms...,spam
2873,See you there!,ham


In [4]:
# unpack into features (X) and labels (y)
# this is NOT a new split - same train_df/test_df from above, just separated by column
X_train, y_train = train_df["Message"], train_df["Class"]
X_test, y_test = test_df["Message"], test_df["Class"]

## Part 2: Feature Engineering

### Preprocessing

Tokenization, lowercasing, stopwords removal, and stemming are preprocessing - they clean up the raw text but don't yet turn it into numeric features. The actual feature engineering (the Featurizer) comes after this, in the Bag-of-Words step below.

### Step 1 - Tokenization (Atomic Segmentation)

split each message into individual word tokens using a regex that matches only word characters (`\w+`). Punctuation is not captured by `\w+`, so it's dropped automatically as part of this step.

In [5]:
import re  # regular expressions module, used for tokenizing text

def tokenize(text):
    # \w+ matches runs of letters/digits/underscore -> punctuation is simply never captured
    return re.findall(r"\w+", text)  # returns a list of the matched token strings, e.g. ["Ok", "i", "am", ...]

# .apply() calls tokenize() once for every message in the Series, and collects all the returned lists into a new Series
X_train_tokens = X_train.apply(tokenize)
X_test_tokens = X_test.apply(tokenize)

# X_train_tokens and X_test_tokens are pandas Series, where each element is a list of tokens (not a single string anymore)
X_train_tokens.head()

82            [Ok, i, am, on, the, way, to, home, hi, hi]
1328    [Ur, balance, is, now, 500, Ur, next, question...
577     [I, m, tired, of, arguing, with, you, about, t...
656     [Tell, them, the, drug, dealer, s, getting, im...
317     [Hmmm, Guess, we, can, go, 4, kb, n, power, yo...
Name: Message, dtype: object

### Step 2 - Lowercase

lowercase every token so that "Free" and "free" for example, will be treated as the same feature instead of being counted as two different features.

In [6]:
def lowercase_tokens(tokens):
    return [t.lower() for t in tokens]  # .lower() converts a string to lowercase letters. list comprehension applies it to every token in the list.

# Same X_train_tokens and X_test_tokens, but now all tokens are lowercase
X_train_tokens = X_train_tokens.apply(lowercase_tokens)
X_test_tokens = X_test_tokens.apply(lowercase_tokens)

X_train_tokens.head()

82            [ok, i, am, on, the, way, to, home, hi, hi]
1328    [ur, balance, is, now, 500, ur, next, question...
577     [i, m, tired, of, arguing, with, you, about, t...
656     [tell, them, the, drug, dealer, s, getting, im...
317     [hmmm, guess, we, can, go, 4, kb, n, power, yo...
Name: Message, dtype: object

### Step 3 - Stopwords removal
remove common words that don't carry much meaning, like "the", "is", "and", etc.

In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS # importing a known list of English stop words from sklearn

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS] # returns a list of tokens that are not in ENGLISH_STOP_WORDS

# Same X_train_tokens and X_test_tokens, but now all stop words are removed
X_train_tokens = X_train_tokens.apply(remove_stopwords)
X_test_tokens = X_test_tokens.apply(remove_stopwords)

X_train_tokens.head()

82                                [ok, way, home, hi, hi]
1328    [ur, balance, 500, ur, question, sang, uptown,...
577             [m, tired, arguing, week, week, want, ll]
656           [tell, drug, dealer, s, getting, impatient]
317     [hmmm, guess, 4, kb, n, power, yoga, haha, dun...
Name: Message, dtype: object

### Step 4 - Stemming

reduce each word to it's base form (for example: "winning", "won", "wins" -> "win"), so different forms of the same word are treated as one feature instead of separate ones.

In [8]:
from nltk.stem import PorterStemmer # importing the PorterStemmer class from the nltk.stem module, which is used for stemming words

stemmer = PorterStemmer() # create a stemmer object that can be used to stem words

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]  # returns a list of stemmed tokens. Porter can produce non-dictionary stems (like: "arguing" -> "argu"), but this doesn't hurt classification since the mapping is consistent across train/test

# Same X_train_tokens and X_test_tokens, but now all tokens are stemmed
X_train_tokens = X_train_tokens.apply(stem_tokens)
X_test_tokens = X_test_tokens.apply(stem_tokens)

X_train_tokens.head()

82                                [ok, way, home, hi, hi]
1328    [ur, balanc, 500, ur, question, sang, uptown, ...
577                 [m, tire, argu, week, week, want, ll]
656                  [tell, drug, dealer, s, get, impati]
317     [hmmm, guess, 4, kb, n, power, yoga, haha, dun...
Name: Message, dtype: object

### Featurizer

This is the actual feature engineering step. turning the processed tokens to numeric Bag-of-Words matrix: every unique word across the training messages becomes a feature (column), and each message (row) gets the raw count of how many times each word appears in it.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer # importing the CountVectorizer class from sklearn, which is used to convert a collection of text documents to a matrix of token counts

vectorizer = CountVectorizer(tokenizer=lambda tokens: tokens, preprocessor=lambda tokens: tokens) # create a CountVectorizer object that uses the tokens i created above, instead of the default behavior of tokenizing and preprocessing 
X_train_bow = vectorizer.fit_transform(X_train_tokens) # learns the vocabulary from the training tokens and returns the train Bag-of-Words matrix (counts)
X_test_bow = vectorizer.transform(X_test_tokens) # applies the vocabulary already learned from train to the test tokens (no re-fitting), returning the test Bag-of-Words matrix

X_train_bow = X_train_bow.toarray() # convert the scipy sparse matrix to a dense numpy array - simpler to index/do math on for the Naive Bayes implementation later
X_test_bow = X_test_bow.toarray()   # same conversion for the test matrix

C:\Users\aviv\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [10]:
import pandas as pd

# displaying the feature engineered data on 2-3 example rows, with actual word names as column headers
print("Train examples:")
display(pd.DataFrame(X_train_bow[:3, 6200:6220], columns=vectorizer.get_feature_names_out()[6200:6220]))  # this range includes "wont", chosen so the example isn't all zeros

print("Test examples:")
display(pd.DataFrame(X_test_bow[:3, 6200:6220], columns=vectorizer.get_feature_names_out()[6200:6220]))  # same range - "wont" is 1 in the test example, proving the counts actually work

Train examples:


,wlcome,wld,wml,wn,wnevr,wnt,wo,woah,wocay,woke,woken,woman,women,won,wondar,wondarful,wonder,wont,woo,woodland
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Test examples:


,wlcome,wld,wml,wn,wnevr,wnt,wo,woah,wocay,woke,woken,woman,women,won,wondar,wondarful,wonder,wont,woo,woodland
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


## Part 3: Naive Bayes Algorithm Implementation
Implementing Multinomial Naive Bayes from scratch, trained on the Count Bag-of-Words features (`X_train_bow`) built in Part 2, to classify each message as spam or ham.

In [11]:
import numpy as np

In [12]:
class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing parameter
        self.classes = np.array([])  # array of unique class labels (e.g., ["ham", "spam"]), set in fit()
        self.priors = {}  # dictionary to hold prior probabilities for each class (e.g., {"ham": 0.8, "spam": 0.2})
        self.word_counts = {}  # dictionary to hold word counts for each class (e.g., {"ham": np.array([...]), "spam": np.array([...])})
        self.vocabulary_size = 0  # total number of unique words in the vocabulary

        # dictionary mapping each class to an array of per-word likelihoods (with Laplace smoothing), one value per vocabulary word
        # e.g., {"ham": np.array([...]), "spam": np.array([...])}, where likelihoods["ham"][i] = P(word_i | ham)
        self.likelihoods = {}

    def fit(self, X_train, y_train):
        self.classes = np.unique(y_train)

        # Calculate class priors
        self.priors = {c: np.sum(y_train == c) / len(y_train) for c in self.classes}  # prior probability for each class
        # Calculate word counts for each class
        for c in self.classes:
            rows_for_class = X_train[y_train == c]  # select rows corresponding to class c
            self.word_counts[c] = np.sum(rows_for_class, axis=0)  # sum the counts of each word across all messages in class c

        # number of unique words (features) in the training data
        self.vocabulary_size = X_train.shape[1]  

        # Calculate likelihoods for each word in each class with Laplace smoothing
        for c in self.classes:
            total_words_in_class = np.sum(self.word_counts[c])  # total word count across all messages in class c
            smoothed_counts = self.word_counts[c] + self.alpha  # NumPy broadcasting: adds alpha to every element of the array at once (no loop needed), e.g. np.array([3, 2, 0]) + 1 -> np.array([4, 3, 1])
            self.likelihoods[c] = smoothed_counts / (total_words_in_class + self.alpha * self.vocabulary_size)

    def predict(self, X_test):
        predictions = [] # will contain the predicted class for each test message e.g., ["ham", "ham", "spam", ...]

        # Iterating the rows of the test matrix, where each row is a message represented as a bag-of-words vector
        for row in X_test:
            log_posteriors = {} # will contain the log posterior probability for each class for the current test message, e.g., {"ham": -12.3, "spam": -15.6}
            for c in self.classes: # 2 iterations: one for "ham" and one for "spam"
                # Calculate log posterior probability for class c
                log_prior = np.log(self.priors[c])  # log(P(c))
                log_likelihood = np.sum(row * np.log(self.likelihoods[c]))  # sum of log(P(word_i | c)) for all words in the message
                log_posteriors[c] = log_prior + log_likelihood  # log(P(c | x)) ∝ log(P(c)) + sum(log(P(word_i | c)))

            # Choose the class with the highest log posterior probability
            predicted_class = max(log_posteriors, key=log_posteriors.get) # Select the class with the highest log posterior (e.g., "ham" if log_posteriors["ham"] > log_posteriors["spam"])
            predictions.append(predicted_class) # Append the predicted class for the current test message to the predictions list

        return np.array(predictions)  # return predictions as a NumPy array e.g., np.array(["ham", "ham", "spam", ...])

## Part 6 (Bonus): Hyperparameter Tuning - Grid Search + 5-Fold CV
Grid search tests different values of the alpha hyperparameter to find the one that gives the best performance. For each value, 5-fold cross-validation is used to evaluate the model on different train/validation splits, and the average F1 score is calculated. The alpha with the highest average F1 score is selected as the best value.

In [13]:
from sklearn.model_selection import KFold  # utility to generate train/validation index splits for k-fold CV
from sklearn.metrics import f1_score        # computes F1 score given true labels and predicted labels
import pandas as pd

alpha_values = [0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 10.0, 20.0]  # the hyperparameter values we want to try (grid search "grid")
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # will split X_train_bow into 5 folds; shuffle so folds aren't just sequential rows; random_state = reproducible shuffle

results = []  # will hold one row per alpha value: {alpha, avg_f1}
for alpha in alpha_values:  # outer loop = grid search over hyperparameter values
    fold_scores = []  # F1 scores from each of the 5 folds, for this alpha

    for train_idx, validation_idx in kf.split(X_train_bow):
        # kf.split() yields 5 pairs of (train_idx, validation_idx) arrays - row-index positions, not labels/values
        # each iteration = 1 fold: train_idx = the 4/5 of rows to train on, validation_idx = the held-out 1/5 to validate on

        X_fold_train, X_fold_validation = X_train_bow[train_idx], X_train_bow[validation_idx]
        # numpy array -> plain [idx] indexing selects those rows

        y_fold_train, y_fold_validation = y_train.iloc[train_idx], y_train.iloc[validation_idx]
        # y_train is a pandas Series -> .iloc[idx] selects rows by position (not by its original index labels like 82, 1328, etc.)

        model = NaiveBayes(alpha=alpha)  # fresh model instance for this fold, using the current alpha being tested
        model.fit(X_fold_train, y_fold_train)  # train only on this fold's training portion
        preds = model.predict(X_fold_validation)      # predict on this fold's held-out portion

        score = f1_score(y_fold_validation, preds, pos_label="spam")
        # pos_label="spam" -> F1 computed only for the spam class (matches the rubric's binary/central-class F1 requirement)

        fold_scores.append(score)  # store this fold's score before moving to the next fold

    results.append({"alpha": alpha, "avg_f1": sum(fold_scores) / len(fold_scores)})
    # after all 5 folds done for this alpha, average their scores -> one number representing this alpha's performance

results_df = pd.DataFrame(results)  # turn the list of {alpha, avg_f1} dicts into a table, one row per alpha
best_row = results_df.loc[results_df["avg_f1"].idxmax()]  # idxmax() finds the row-index with the highest avg_f1; .loc[] retrieves that full row as a Series e.g., alpha=3.0, avg_f1=0.93

display(results_df)
print("Best alpha:", best_row["alpha"], "with avg F1:", best_row["avg_f1"])

,alpha,avg_f1
0,0.1,0.915828
1,0.5,0.919379
2,1.0,0.924110
3,2.0,0.933621
4,3.0,0.934152
5,4.0,0.924288
6,5.0,0.920995
7,10.0,0.889611
8,20.0,0.836872


Best alpha: 3.0 with avg F1: 0.9341516128299409


## Part 4: Training
Using the best alpha identified in the bonus grid search above, the final Naive Bayes model is trained on the entire training set. This allows the model to use all available training data before being evaluated on the test set.

In [14]:
nb_model = NaiveBayes(alpha=best_row["alpha"])  # instantiate the final model using the best alpha found by the grid search above
nb_model.fit(X_train_bow, y_train)  # train the final model on the full training set (not just a fold)

## Part 5: Prediction and Model Quality Evaluation on the Test Set
Using the final model trained in Part 4, predict on the test set and evaluate quality using F1 score on the spam class.

In [22]:
preds = nb_model.predict(X_test_bow)  # predict on the test set using the final model
score = f1_score(y_test, preds, pos_label="spam")  # compute the F1 score on the test set

print("First 5 true labels:")
display(y_test.head(5))  # display the first 5 true labels
print()

preds = pd.Series(preds, name="Prediction", index=y_test.index)   
print("First 5 predictions:")
display(preds.head(5))  # display the first 5 predictions to compare with the true labels

print("Final model F1 score on test set:", score)

First 5 true labels:


3625     ham
4570     ham
1192     ham
673     spam
2873     ham
Name: Class, dtype: object


First 5 predictions:


3625     ham
4570     ham
1192     ham
673     spam
2873     ham
Name: Prediction, dtype: object

Final model F1 score on test set: 0.9219858156028369
